# Week 3 Homework

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

In [3]:
predict_conversion_df = pd.read_csv("digital_marketing_campaign_dataset.csv")
google_ads_df = pd.read_csv("GoogleAds_DataAnalytics_Sales_Uncleaned.csv")
marketing_products_df = pd.read_csv("marketing_and_product_performance.csv")

## Backward Selection for Regression of predict_conversion_df

In [4]:
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import SequentialFeatureSelector

# Data Preparation
df = predict_conversion_df.drop(columns=['CustomerID'])

# Convert categorical features into dummy/indicator variables
X = pd.get_dummies(df.drop(columns=['Conversion']), drop_first=True)
y = df['Conversion']

# Set up Linear Regression and Backward Selection
model = LinearRegression()
# n_features_to_select='auto' will select half of the features by default.
sfs = SequentialFeatureSelector(model, direction='backward', scoring='r2', cv=5)

# Fit the Backward Selection model
print("Running backward selection (this might take a moment)...")
sfs.fit(X, y)

# Display Results
selected_features = X.columns[sfs.get_support()]
print(f"Original number of features: {X.shape[1]}")
print(f"Selected number of features: {len(selected_features)}\n")

print("Selected features:")
for feature in selected_features:
    print(f"- {feature}")

# Fit final model with the selected features
model.fit(X[selected_features], y)
print(f"\nFinal Model R-squared: {model.score(X[selected_features], y):.4f}")

Running backward selection (this might take a moment)...
Original number of features: 21
Selected number of features: 11

Selected features:
- AdSpend
- ClickThroughRate
- ConversionRate
- WebsiteVisits
- PagesPerVisit
- TimeOnSite
- EmailOpens
- EmailClicks
- PreviousPurchases
- LoyaltyPoints
- CampaignType_Conversion

Final Model R-squared: 0.1387


## PLSR of predict_conversion_df

In [6]:
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import cross_val_predict
import numpy as np

# Scaling the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Iterate through components to find optimal number using 5-fold CV
components = np.arange(1, X.shape[1] + 1)
cv_r2_scores = []

for i in components:
    pls = PLSRegression(n_components=i)
    y_cv = cross_val_predict(pls, X_scaled, y, cv=5)
    r2 = r2_score(y, y_cv)
    cv_r2_scores.append(r2)

optimal_components = components[np.argmax(cv_r2_scores)]
best_cv_r2 = max(cv_r2_scores)

print(f"Optimal number of PLS components: {optimal_components}")
print(f"Best Cross-Validated R-squared: {best_cv_r2:.4f}\n")

# Fit the final PLS model with the optimal number of components
pls_final = PLSRegression(n_components=optimal_components)
pls_final.fit(X_scaled, y)

print(f"Training R-squared (with {optimal_components} components): {pls_final.score(X_scaled, y):.4f}")

Optimal number of PLS components: 4
Best Cross-Validated R-squared: 0.1044

Training R-squared (with 4 components): 0.1396


## Backward Selection for Regression of google_ads_df

In [5]:
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import RFE
from sklearn.preprocessing import StandardScaler
import pandas as pd

# 1. Data Preparation
df_ads = google_ads_df.copy()

# Convert 'Conversion Rate' if it's a string, drop otherwise target-leaking fields
if df_ads['Conversion Rate'].dtype == object:
    df_ads['Conversion Rate'] = df_ads['Conversion Rate'].str.rstrip('%').astype('float') / 100.0

cols_to_drop = ['Ad_ID', 'Campaign_Name', 'Ad_Date', 'Keyword', 'Conversion Rate', 'Sale_Amount']
df_ads = df_ads.drop(columns=[c for c in cols_to_drop if c in df_ads.columns])
df_ads = df_ads.dropna()

# Convert categorical features into dummy variables
X_ads = pd.get_dummies(df_ads.drop(columns=['Conversions']), drop_first=True)
y_ads = df_ads['Conversions']

# Scale features (Helps Linear Regression coefficient stability)
scaler = StandardScaler()
X_ads_scaled = pd.DataFrame(scaler.fit_transform(X_ads), columns=X_ads.columns)

# 2. Set up Linear Regression and Recursive Feature Elimination (RFE)
# RFE is a much faster backward selection approach for Linear Models
model_ads = LinearRegression()
# This removes the least important feature one by one until half are left
rfe = RFE(estimator=model_ads, n_features_to_select=X_ads_scaled.shape[1] // 2, step=1)

# 3. Fit the Backward Selection model
print("Running faster backward selection (RFE) for google_ads_df...")
rfe.fit(X_ads_scaled, y_ads)

# 4. Display Results
selected_features_ads = X_ads.columns[rfe.support_]
print(f"Original number of features: {X_ads.shape[1]}")
print(f"Selected number of features: {len(selected_features_ads)}\n")

print("Selected features:")
for feature in selected_features_ads:
    print(f"- {feature}")

# Fit final model with the selected features
model_ads.fit(X_ads_scaled[selected_features_ads], y_ads)
print(f"\nFinal Model R-squared: {model_ads.score(X_ads_scaled[selected_features_ads], y_ads):.4f}")

Running faster backward selection (RFE) for google_ads_df...
Original number of features: 1930
Selected number of features: 965

Selected features:
- Clicks
- Impressions
- Cost_$180.04
- Cost_$180.05
- Cost_$180.06
- Cost_$180.25
- Cost_$180.42
- Cost_$180.49
- Cost_$180.5
- Cost_$180.57
- Cost_$180.58
- Cost_$180.63
- Cost_$180.64
- Cost_$180.66
- Cost_$180.7
- Cost_$180.74
- Cost_$180.82
- Cost_$180.88
- Cost_$181.02
- Cost_$181.11
- Cost_$181.12
- Cost_$181.18
- Cost_$181.22
- Cost_$181.34
- Cost_$181.46
- Cost_$181.48
- Cost_$181.56
- Cost_$181.58
- Cost_$181.7
- Cost_$181.74
- Cost_$181.87
- Cost_$181.92
- Cost_$181.95
- Cost_$181.96
- Cost_$182.15
- Cost_$182.27
- Cost_$182.5
- Cost_$182.54
- Cost_$182.57
- Cost_$182.84
- Cost_$182.85
- Cost_$182.9
- Cost_$182.98
- Cost_$182.99
- Cost_$183.05
- Cost_$183.12
- Cost_$183.15
- Cost_$183.16
- Cost_$183.22
- Cost_$183.25
- Cost_$183.26
- Cost_$183.39
- Cost_$183.4
- Cost_$183.49
- Cost_$183.52
- Cost_$183.57
- Cost_$183.62
- Cost_$18

## PLSR Analysis of google_ads_df

In [7]:
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import cross_val_predict
import numpy as np

# We'll use the 'X_ads_scaled' and 'y_ads' already prepared and scaled
# To avoid excessive computation time with 1930 features, we cap max components to test at 20
max_comps_to_test = min(20, X_ads_scaled.shape[1])
components_ads = np.arange(1, max_comps_to_test + 1)
cv_r2_scores_ads = []

print("Running PLSR cross-validation...")
for i in components_ads:
    pls_ads = PLSRegression(n_components=i)
    # Get cross-validated predictions to calculate a reliable R-squared
    y_cv_ads = cross_val_predict(pls_ads, X_ads_scaled, y_ads, cv=5)
    r2_ads = r2_score(y_ads, y_cv_ads)
    cv_r2_scores_ads.append(r2_ads)

optimal_components_ads = components_ads[np.argmax(cv_r2_scores_ads)]
best_cv_r2_ads = max(cv_r2_scores_ads)

print(f"Optimal number of PLS components (tested up to {max_comps_to_test}): {optimal_components_ads}")
print(f"Best Cross-Validated R-squared: {best_cv_r2_ads:.4f}\n")

# Fit the final PLS model with the optimal number of components
pls_ads_final = PLSRegression(n_components=optimal_components_ads)
pls_ads_final.fit(X_ads_scaled, y_ads)

print(f"Training R-squared (with {optimal_components_ads} components): {pls_ads_final.score(X_ads_scaled, y_ads):.4f}")

Running PLSR cross-validation...
Optimal number of PLS components (tested up to 20): 1
Best Cross-Validated R-squared: -0.1523

Training R-squared (with 1 components): 0.8537


## Backward Selection for Regression of marketing_products_df

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import RFE
from sklearn.preprocessing import StandardScaler
import pandas as pd

# 1. Data Preparation
df_mkt = marketing_products_df.copy()

# Drop identifier columns and columns
cols_to_drop_mkt = ['Campaign_ID', 'Product_ID', 'Customer_ID', 'Flash_Sale_ID', 'Bundle_ID', 'Common_Keywords', 
                    'Revenue_Generated', 'ROI', 'Units_Sold']
df_mkt = df_mkt.drop(columns=[c for c in cols_to_drop_mkt if c in df_mkt.columns])
df_mkt = df_mkt.dropna()

# Convert remaining categorical features into dummy variables
X_mkt = pd.get_dummies(df_mkt.drop(columns=['Conversions']), drop_first=True)
y_mkt = df_mkt['Conversions']

# Scale features
scaler_mkt = StandardScaler()
X_mkt_scaled = pd.DataFrame(scaler_mkt.fit_transform(X_mkt), columns=X_mkt.columns)

# 2. Set up Linear Regression and RFE 
model_mkt = LinearRegression()
rfe_mkt = RFE(estimator=model_mkt, n_features_to_select=X_mkt_scaled.shape[1] // 2, step=1)

# 3. Fit the model
print("Running backward selection (RFE) for marketing_products_df...")
rfe_mkt.fit(X_mkt_scaled, y_mkt)

# 4. Display Results
selected_features_mkt = X_mkt.columns[rfe_mkt.support_]
print(f"Original number of features: {X_mkt.shape[1]}")
print(f"Selected number of features: {len(selected_features_mkt)}\n")

print("Selected features:")
for feature in selected_features_mkt:
    print(f"- {feature}")

# Fit final model with the selected features
model_mkt.fit(X_mkt_scaled[selected_features_mkt], y_mkt)
print(f"\nFinal Model R-squared: {model_mkt.score(X_mkt_scaled[selected_features_mkt], y_mkt):.4f}")

Running backward selection (RFE) for marketing_products_df...
Original number of features: 8
Selected number of features: 4

Selected features:
- Budget
- Discount_Level
- Bundle_Price
- Subscription_Tier_Standard

Final Model R-squared: 0.0014


## PCR vs PLSR for marketing_products_df

Both PCR and PLSR do not seem to be a good fit for analyzing marketing_products_df. The baseline R-squared is nearly zero and there's not enough of a correlation.

In [ ]:
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score

# Max components to test is limited by the number of remaining features
n_comps_mkt = min(5, X_mkt_scaled.shape[1])

# 1. Partial Least Squares Regression (PLSR)
plsr_mkt = PLSRegression(n_components=n_comps_mkt)
plsr_mkt.fit(X_mkt_scaled, y_mkt)
plsr_score_mkt = plsr_mkt.score(X_mkt_scaled, y_mkt)

# 2. Principal Component Regression (PCR)
pcr_mkt = make_pipeline(PCA(n_components=n_comps_mkt), LinearRegression())
pcr_mkt.fit(X_mkt_scaled, y_mkt)
pcr_score_mkt = pcr_mkt.score(X_mkt_scaled, y_mkt)

print(f"PLSR R-squared ({n_comps_mkt} components): {plsr_score_mkt:.4f}")
print(f"PCR R-squared ({n_comps_mkt} components):  {pcr_score_mkt:.4f}")

PLSR R-squared (5 components): 0.0016
PCR R-squared (5 components):  0.0014
